In [ ]:
# ============================================================
#  TEXT → SQL GENERATOR  |  Generative AI (NLP)  |  JupyterLab
#  Paste this entire cell and run it (Shift+Enter)
# ============================================================

# ---------- 0. Install dependencies (runs once) ----------
import subprocess, sys
for pkg in ["anthropic", "ipywidgets", "IPython"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import anthropic, textwrap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- 1. Configure Anthropic client ----------
import os
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")   # <-- set env var OR paste key here
if not API_KEY:
    API_KEY = input("Enter your Anthropic API key: ").strip()
client = anthropic.Anthropic(api_key=API_KEY)

# ---------- 2. Sample database schema ----------
SCHEMA = textwrap.dedent("""
    Database: ecommerce_db
    Tables:
      users       (id INT PK, name VARCHAR, email VARCHAR, country VARCHAR, created_at TIMESTAMP)
      orders      (id INT PK, user_id INT FK→users.id, status VARCHAR, total_amount DECIMAL, created_at TIMESTAMP)
      products    (id INT PK, name VARCHAR, category VARCHAR, price DECIMAL, stock INT)
      order_items (id INT PK, order_id INT FK→orders.id, product_id INT FK→products.id, quantity INT, unit_price DECIMAL)
""").strip()

SYSTEM_PROMPT = f"""\
You are an expert SQL engineer. Convert natural-language questions into clean, optimized SQL queries.

{SCHEMA}

Rules:
- Output ONLY the SQL query (no markdown fences, no preamble).
- Use standard PostgreSQL/MySQL-compatible SQL.
- Use table aliases and readable formatting (uppercase keywords, newlines).
- Add brief inline comments (--) for any complex logic.
- After the SQL, on a new line starting with "-- Explanation:", add a 1-sentence plain-English summary.
"""

# ---------- 3. Core NL→SQL function ----------
def text_to_sql(natural_language_query: str, stream: bool = True) -> str:
    """
    Convert a natural-language question to SQL using Claude.
    Returns the generated SQL string.
    """
    if stream:
        sql_parts = []
        with client.messages.stream(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": natural_language_query}],
        ) as s:
            for chunk in s.text_stream:
                print(chunk, end="", flush=True)
                sql_parts.append(chunk)
        print()  # final newline
        return "".join(sql_parts)
    else:
        msg = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": natural_language_query}],
        )
        return msg.content[0].text

# ---------- 4. Interactive Widget UI ----------
EXAMPLES = [
    "Top 5 customers by total spending in the last 30 days with their email",
    "Monthly revenue totals for each month in the last 12 months",
    "All products with stock less than 10, sorted by stock ascending",
    "Pending orders older than 7 days with customer name and order total",
    "Average order value per country, only countries with more than 5 orders",
    "Most popular product categories by number of items sold",
]

# --- Widgets ---
style = {"description_width": "initial"}
layout_full = widgets.Layout(width="100%")

title_html = widgets.HTML(
    value="<h2 style='margin:0 0 4px;font-size:20px;'>🔍 Text → SQL Generator</h2>"
           "<p style='color:#555;font-size:13px;margin:0 0 12px;'>Powered by Claude · Natural Language Processing</p>"
)

schema_html = widgets.HTML(
    value=f"<details><summary style='cursor:pointer;font-weight:500;font-size:13px;'>📋 Schema ({SCHEMA.splitlines()[0]})</summary>"
           f"<pre style='background:#f6f8fa;padding:10px;border-radius:6px;font-size:12px;'>{SCHEMA}</pre></details>"
)

nl_input = widgets.Textarea(
    placeholder="Describe the data you want in plain English…\ne.g. Show me the top 5 customers by total spending last month",
    layout=widgets.Layout(width="100%", height="80px"),
    style=style,
)

example_dropdown = widgets.Dropdown(
    options=[""] + EXAMPLES,
    value="",
    description="Quick examples:",
    layout=widgets.Layout(width="100%"),
    style=style,
)

stream_toggle = widgets.Checkbox(value=True, description="Stream output", style=style)
generate_btn = widgets.Button(description="⚡ Generate SQL", button_style="primary",
                               layout=widgets.Layout(width="160px"))
clear_btn    = widgets.Button(description="🗑 Clear", button_style="",
                               layout=widgets.Layout(width="100px"))
copy_hint    = widgets.HTML(value="")
output_area  = widgets.Output(layout=widgets.Layout(
    border="1px solid #d0d7de", padding="12px",
    border_radius="8px", min_height="60px", width="100%"
))

last_sql = {"value": ""}

def on_example_change(change):
    if change["new"]:
        nl_input.value = change["new"]
        example_dropdown.value = ""

def on_generate(b):
    query = nl_input.value.strip()
    if not query:
        with output_area:
            clear_output()
            print("⚠️  Please enter a natural-language query first.")
        return
    generate_btn.disabled = True
    generate_btn.description = "⏳ Generating…"
    copy_hint.value = ""
    with output_area:
        clear_output()
        print(f"Query: {query}\n" + "─" * 60)
        sql = text_to_sql(query, stream=stream_toggle.value)
        last_sql["value"] = sql
        copy_hint.value = (
            "<span style='font-size:12px;color:#666;'>"
            "✅ SQL generated — copy from the cell above or use <code>last_sql['value']</code>"
            "</span>"
        )
    generate_btn.disabled = False
    generate_btn.description = "⚡ Generate SQL"

def on_clear(b):
    nl_input.value = ""
    with output_area:
        clear_output()
    copy_hint.value = ""
    last_sql["value"] = ""

example_dropdown.observe(on_example_change, names="value")
generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)

btn_row = widgets.HBox([generate_btn, clear_btn, stream_toggle],
                        layout=widgets.Layout(gap="8px", align_items="center"))

ui = widgets.VBox([
    title_html,
    schema_html,
    widgets.HTML(value="<b style='font-size:13px;'>Natural Language Query</b>"),
    nl_input,
    example_dropdown,
    btn_row,
    output_area,
    copy_hint,
], layout=widgets.Layout(width="100%", gap="8px"))

display(ui)

# ---------- 5. Also usable programmatically ----------
# Uncomment below to run without the widget UI:
# sql = text_to_sql("Get total revenue per product category this year")
# print(sql)
